# 10. 음성 — 소리는 숫자다 · 받아쓰기(Whisper) · 읽어 주기(Supertonic)

> 사진이 `(세로, 가로, 3)` 숫자 배열이었다면, 소리는 **시간 순서대로 늘어선 숫자 한 줄**이다.
> 그 숫자를 글로(받아쓰기), 글을 그 숫자로(읽어 주기) 바꾸는 모델을 쓴다.

### 이 시간에 배우는 것
1. 소리 = 표본율(1초에 몇 번 재나) · −1~1 실수 배열, 파형 보기 · 소리 크기(RMS)
2. **받아쓰기(STT)**: Whisper small (INT8) — `openvino_genai.WhisperPipeline`
3. **읽어 주기(TTS)**: Supertonic — ONNX 모델 **4개**를 onnxruntime 으로 차례로
4. 실시간 배수(RTF): 1초 분량을 몇 초에 만드나

### 읽어 주기 안에서
```
글 ─▶ [길이 예측] 몇 초짜리 말인가 ─▶ [글 인코딩] 글의 뜻 ─▶ [잡음 제거 × steps 번] 잡음에서 말소리 모양을 깎아 냄 ─▶ [보코더] 소리 숫자로
       duration_predictor              text_encoder          vector_estimator (확산 모델)                         vocoder
```
코드는 프로그램 폴더의 `speech_routes.py` 의 `_Supertonic` — 열어서 읽어 보자.

## 1. 준비

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

import sys
import sounddevice as sd
from IPython.display import Audio
import matplotlib.pyplot as plt     # 그래프 글자는 영어로

SR = 16000                          # 표본율: 1초에 16000번 잰다 (Whisper 가 원하는 값)

## 2. 녹음 — 4초
`sd.rec` 는 녹음을 시작만 하고 바로 돌아온다. `sd.wait()` 로 끝날 때까지 기다린다.
결과는 `(표본 수, 채널 수)` 배열 → 한 줄로 편다. 4초 × 16000 = 64000 개 숫자.

In [ ]:
SECONDS = 4
print("말하세요 ...")
rec = sd.rec(int(SECONDS * SR), samplerate=SR, channels=1, dtype="float32")
sd.wait()
audio = rec.reshape(-1)
rms = float(np.sqrt((audio ** 2).mean()))           # 소리 크기(평균 에너지)
print(f"표본 {len(audio)}개 = {len(audio) / SR:.1f}초 · 가장 큰 값 {np.abs(audio).max():.2f} · RMS {rms:.3f}")
print("RMS 가 0.01 보다 작으면 거의 조용한 것 — 마이크를 확인")
display(Audio(audio, rate=SR))

In [ ]:
t = np.arange(len(audio)) / SR
plt.figure(figsize=(8, 2)); plt.plot(t, audio, lw=0.5); plt.xlabel("seconds"); plt.ylim(-1, 1); plt.title("waveform"); plt.show()

## 3. 받아쓰기 — Whisper
Whisper 는 소리를 **멜 스펙트로그램**(시간 × 주파수 그림)으로 바꿔 인코더에 넣고, 디코더가 글 토큰을 하나씩 만든다 (VLM 과 같은 생성 방식).
`language="<|ko|>"` 로 언어를 정해 주면 더 정확하고 빠르다.

In [ ]:
import openvino_genai as og

STT_DIR = MODELS / "stt"
DEVICE = "GPU" if any(d.startswith("GPU") for d in core.available_devices) else "CPU"
t0 = time.perf_counter()
whisper = og.WhisperPipeline(str(STT_DIR), DEVICE)
print(f"올림 {time.perf_counter() - t0:.1f}초 ({DEVICE})")

def stt(a, lang="ko"):
    return str(whisper.generate(a, language=f"<|{lang}|>", task="transcribe")).strip()

t0 = time.perf_counter()
text = stt(audio)
print(f"받아쓰기: {text}   ({(time.perf_counter() - t0) * 1000:.0f} ms)")

## 4. 읽어 주기 — Supertonic (ONNX 4개)

In [ ]:
sys.path.insert(0, str(ROOT))                      # 프로그램 폴더의 speech_routes.py 를 읽는다
import speech_routes

tts = speech_routes._Supertonic(speech_routes.TTS_DIR)
for name, sess in [("길이 예측", tts.dp), ("글 인코딩", tts.enc), ("잡음 제거", tts.est), ("보코더", tts.voc)]:
    print(f"{name:6s} 입력 {[i.name for i in sess.get_inputs()]}")
print("목소리:", speech_routes.VOICES, "| 표본율:", tts.sr)

In [ ]:
def say(text, voice="F1", steps=8, speed=1.05, play=True):
    t0 = time.perf_counter()
    wav = tts.say(text, lang="ko", voice=voice, steps=steps, speed=speed)
    dt = time.perf_counter() - t0
    print(f"{len(wav) / tts.sr:.1f}초 분량을 {dt:.2f}초에 만듦 (실시간의 {len(wav) / tts.sr / dt:.1f}배)")
    if play:
        display(Audio(wav, rate=tts.sr))
    return wav

wav = say("안녕하세요. 이 목소리는 이 컴퓨터 안에서 만들었어요.")

In [ ]:
# steps(잡음 제거 반복 횟수)에 따라 속도가 어떻게 바뀌나
for st in (2, 4, 8, 16):
    print(f"steps={st:2d}: ", end="")
    say("반복 횟수를 바꿔 봅니다.", steps=st, play=False)

## 자주 나는 오류
| 증상 | 원인 · 고침 |
|---|---|
| `PortAudioError` | 마이크가 없거나 다른 프로그램이 쓰는 중. 우분투는 `libportaudio2` |
| 받아쓰기가 빈 글 | 너무 조용함(RMS 확인) · 너무 짧음 |
| 엉뚱한 언어로 받아씀 | `language` 를 정해 준다 |
| `speech_routes` import 실패 | `EDGE_ROOT` 가 프로그램 폴더를 가리키는지 (`lab.bat` 으로 켰는지) |

## 정리
- 소리 = 표본율 × 시간 개의 실수. 크기는 RMS 로 잰다.
- STT 와 VLM 은 같은 "토큰을 하나씩 만드는" 생성 방식이다.
- TTS(Supertonic)는 모델 4개의 파이프라인. `steps` 로 속도와 음질을 바꾼다.

---
# 🛠 미니 프로젝트 — 미니 음성 비서

**만들 것**: 말을 듣고(STT) → 명령을 알아보고 → 대답을 말한다(TTS). 3번 주고받는다.

| 말하면 | 대답 |
|---|---|
| "몇 시" 가 들어 있으면 | 지금 시각 |
| "며칠" · "날짜" | 오늘 날짜 |
| "더하기" (예: 3 더하기 5) | 계산 결과 |
| 그 밖 | 들은 말을 따라 하기 |

**쓰는 것**: 녹음(`sd.rec`) · `stt` · `say` · 정규식(`re`)

규칙으로 명령을 고르는 부분을 08/09 의 LLM 으로 바꾸면 "진짜" 비서가 된다 — 더 해 보기 참고.

In [ ]:
import re
from datetime import datetime

def listen(seconds=4):
    print("🎤 듣는 중 ...")
    r = sd.rec(int(seconds * SR), samplerate=SR, channels=1, dtype="float32"); sd.wait()
    return r.reshape(-1)

def reply(heard):
    if "몇 시" in heard or "몇시" in heard:
        n = datetime.now(); return f"지금은 {n.hour}시 {n.minute}분이에요."
    if "며칠" in heard or "날짜" in heard:
        n = datetime.now(); return f"오늘은 {n.month}월 {n.day}일이에요."
    m = re.search(r"(\d+)\s*더하기\s*(\d+)", heard)
    if m:
        a, b = int(m.group(1)), int(m.group(2)); return f"{a} 더하기 {b} 는 {a + b} 예요."
    return f"'{heard}' 라고 하셨어요." if heard else "잘 못 들었어요."

for turn in range(3):
    heard = stt(listen())
    answer_text = reply(heard)
    print(f"[{turn + 1}] 들은 말: {heard!r} → 대답: {answer_text}")
    say(answer_text)

**더 해 보기**
- 대답을 노트북이 아니라 스피커로 바로: `sd.play(wav, tts.sr); sd.wait()`
- "그만" 이라고 하면 끝나게 바꾸기
- 규칙 대신 LLM 에게 대답을 만들게 하기 (09 번의 `pipe.generate(질문)`) — 대신 메모리를 많이 쓴다